In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS main.lakehouse_marketing.bronze;

In [0]:
BASE_PATH = "/Volumes/main/lakehouse_marketing"
RAW_PATH = f"{BASE_PATH}/raw"
BRONZE_PATH = f"{BASE_PATH}/bronze"

* Definição do Schema e letura da RAW

In [0]:
users_schema = StructType([
    StructField("country", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("email", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("user_id", StringType(), True)
])

df_users_raw = spark.read\
                .schema(users_schema)\
                .option("header", "true")\
                .csv(f"{RAW_PATH}/users")

In [0]:
display(df_users_raw.sample(0.001))

* **Metadados para bronze (ingestion + source_file)**

    * Garantir auditoria
    * Debug
    * Reprocesso

In [0]:
df_users_bronze = df_users_raw\
                    .withColumn("ingestion_timestamp", F.current_timestamp())\
                    .withColumn("source_file", F.col('_metadata.file_path'))


* **Escrita**

    * Delta Lake
    * Versionamento
    * ACID
    * Time Travel

In [0]:
BRONZE_USERS_PATH = f"{BRONZE_PATH}/users"

df_users_bronze.write\
    .format("delta")\
    .mode("overwrite")\
    .save(BRONZE_USERS_PATH)

* **Validação**

In [0]:
# Deve bater com as 5000 linhas
spark.read\
    .format("delta")\
    .load("/Volumes/main/lakehouse_marketing/bronze/users/")\
    .count()

In [0]:
display(spark.read\
    .format("delta")\
    .load("/Volumes/main/lakehouse_marketing/bronze/users/"))